In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('qoute_dataset.csv')

In [5]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [6]:
df.shape

(3038, 2)

In [7]:
quotes = df['quote']
import string
quotes=quotes.str.lower()
quotes=quotes.apply(lambda x: x.translate(str.maketrans('', '', string.punctuation)))

In [8]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer


vocab_size = 10000
token = Tokenizer(num_words=vocab_size)
token.fit_on_texts(quotes)

In [20]:
word_index = token.word_index
wprd_index

NameError: name 'wprd_index' is not defined

In [11]:
sequence = token.texts_to_sequences(quotes)

In [12]:
print(sequence[0])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]


In [13]:
X=[]
y=[]

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [14]:
X

[[713],
 [713, 62],
 [713, 62, 29],
 [713, 62, 29, 19],
 [713, 62, 29, 19, 16],
 [713, 62, 29, 19, 16, 946],
 [713, 62, 29, 19, 16, 946, 10],
 [713, 62, 29, 19, 16, 946, 10, 7],
 [713, 62, 29, 19, 16, 946, 10, 7, 5],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104],
 [713,
  62,
  29,
  19,
  16,
  946,
  10,
  7,
  5,
  1156,
  8,
  70,
  293,
  10,
  145,
  12,
  809,
  104,
  752],
 [713,
  62,
  29,
  19,
  16,
  946,
  10,
  7,
  5,
  1156,
  8,
  70,
  293,
  10,
  145,
  12,
  809,
  

In [15]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Calculate the maximum sequence length
max_sequence_len = max([len(x) for x in X])

# Apply padding to X
X_padded = pad_sequences(X, maxlen=max_sequence_len, padding='pre')

print(f"Padded X shape: {X_padded.shape}")

Padded X shape: (85271, 745)


In [16]:
y = np.array(y)
y.shape

(85271,)

In [17]:
from tensorflow.keras.utils import to_categorical
y_onehot = to_categorical(y, num_classes=vocab_size)

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

embedding_dim = 50 # You can adjust this value
rnn_unit = 128

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim, input_length=max_sequence_len))
model.add(LSTM(rnn_unit)) # Using a GRU or SimpleRNN could also be options
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [19]:
epochs = 100
batch_size = 128

history = model.fit(X_padded, y_onehot, epochs=epochs, batch_size=batch_size, validation_split=0.1)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 55ms/step - accuracy: 0.0388 - loss: 6.7502 - val_accuracy: 0.0458 - val_loss: 6.6804
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 54ms/step - accuracy: 0.0576 - loss: 6.3186 - val_accuracy: 0.0659 - val_loss: 6.5586
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.0808 - loss: 6.0440 - val_accuracy: 0.0896 - val_loss: 6.4622
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.0988 - loss: 5.8258 - val_accuracy: 0.0970 - val_loss: 6.4145
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1099 - loss: 5.6392 - val_accuracy: 0.1024 - val_loss: 6.4060
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1197 - loss: 5.4656 - val_accuracy: 0.1082 - val_loss: 6.4120
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1282 - loss: 5.3064 - val_accuracy: 0.1078 - val_loss: 6.4445
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1370 - loss: 5

In [21]:
model.save('lstm_model.h5')

In [25]:
# Create a reverse word index for mapping integer to word
reverse_word_index = dict([(value, key) for (key, value) in token.word_index.items()])

def predict_next_word(seed_text, model, tokenizer, max_sequence_len, reverse_word_index):
    # Preprocess the seed text
    seed_text = seed_text.lower()
    seed_text = seed_text.translate(str.maketrans('', '', string.punctuation))

    # Tokenize the seed text
    sequence = tokenizer.texts_to_sequences([seed_text])[0]

    # Pad the sequence to the maximum length
    padded_sequence = pad_sequences([sequence], maxlen=max_sequence_len, padding='pre')

    # Predict the next word's probabilities
    predicted_probabilities = model.predict(padded_sequence, verbose=0)[0]

    # Get the index of the most probable word
    predicted_word_index = np.argmax(predicted_probabilities)

    # Map the index back to a word
    predicted_word = reverse_word_index.get(predicted_word_index, "")

    return predicted_word

In [34]:
def predict_sentence(seed_text, model, tokenizer, max_sequence_len, reverse_word_index, num_words=10):
  for _ in range(num_words):
    next_word = predict_next_word(seed_text, model, tokenizer, max_sequence_len, reverse_word_index)
    if next_word == '':
      break
    seed_text += ' ' + next_word
  return seed_text

In [36]:
# Example usage:
seed_text = "who is the"
complete_text = predict_sentence(seed_text, model, token, max_sequence_len, reverse_word_index)
print(f"Seed Text: '{seed_text}'")
print(f"Predicted Next Word: '{complete_text}'")

Seed Text: 'who is the'
Predicted Next Word: 'who is the unknown she will rise from a man who is constantly'


In [37]:
import pickle

# Save the tokenizer
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(token, f)

# Save max_sequence_len
with open('max_sequence_len.pkl', 'wb') as f:
    pickle.dump(max_sequence_len, f)

print("Tokenizer and max_sequence_len saved successfully.")

Tokenizer and max_sequence_len saved successfully.
